In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 04 rerun after date-fix

## Contexto y objetivo

Este notebook consolida el rerun completo de Day 04 tras detectar un bug upstream de parseo de fechas en `src/etl/extract/extract_ofertas_raw.py`.

Objetivos:
- comparar `antes vs después` del bug fix,
- validar que `2018 -> 2017` desaparece en la cadena `raw -> typed -> V2`,
- rehacer baseline y Day 04 completo sobre base corregida,
- dejar una decisión final única para cerrar Day 04.


In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == 'notebooks' else PROJECT_ROOT
ARCHIVE_REGISTRY = PROJECT_ROOT / 'artifacts/public/metrics/archive/20260306_pre_date_fix_final_baseline_vs_candidates.csv'
CURRENT_REGISTRY = PROJECT_ROOT / 'artifacts/public/metrics/final_baseline_vs_candidates.csv'
EXTRACT_REPORT = PROJECT_ROOT / 'artifacts/public/etl_dq_ofertas_tabla_extract.json'
BASELINE_NEW = PROJECT_ROOT / 'artifacts/public/metrics/baseline/20260306T_day04_rerun_baseline_baseline_metrics.json'
DAY042_OLD = PROJECT_ROOT / 'artifacts/public/metrics/candidates/20260306/20260306T_day042_train_day042_transport_run_summary.json'
DAY042_NEW = PROJECT_ROOT / 'artifacts/public/metrics/candidates/20260306/20260306T_day04_rerun_day042_train_day042_transport_run_summary.json'
DAY043_OLD = PROJECT_ROOT / 'artifacts/public/metrics/candidates/20260306/20260306T_day043_train_day043_transport_run_summary.json'
DAY043_NEW = PROJECT_ROOT / 'artifacts/public/metrics/candidates/20260306/20260306T_day04_rerun_day043_train_day043_transport_run_summary.json'
DAY041_NEW = PROJECT_ROOT / 'artifacts/public/metrics/candidates/20260306/20260306T_day04_rerun_day041_train_day041_ablation_run_summary.json'
V2_PATH = PROJECT_ROOT / 'data/public/dataset_modelo_proveedor_v2_candidates.csv'
TYPED_PATH = PROJECT_ROOT / 'data/public/support/ofertas_typed.csv'
RAW_MINCOST_PATH = PROJECT_ROOT / 'data/public/support/ofertas_raw_mincost.csv'


## Recap pre-fix vs post-fix en registry

In [3]:
old_registry = pd.read_csv(ARCHIVE_REGISTRY)
new_registry = pd.read_csv(CURRENT_REGISTRY)

cols = ['run_id', 'day_id', 'model_variant', 'top1_hit', 'top2_hit', 'balanced_accuracy', 'coverage', 'promotion_decision']
display(Markdown('### Registry pre-fix'))
display(old_registry[cols])
display(Markdown('### Registry post-fix'))
display(new_registry[cols])


### Registry pre-fix

,run_id,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
0,20260304T120000Z,Day 02,LR_smote_0.5,0.772123,0.862894,0.861463,1.0,keep_baseline
1,20260305T160207Z,Day 03,BASELINE_WITH_DETERMINISTIC_LAYER_PRODUCT_002_...,0.788834,0.862894,0.861463,1.0,promote
2,20260306T084542Z,Day 04,V3_A_LR_smote_0.5_v1,0.739840,0.852640,0.852651,1.0,keep_baseline
3,20260306T084542Z,Day 04,V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLL...,0.766046,0.856817,0.852651,1.0,keep_baseline
4,20260306T095152Z,Day 04.1,V2_SOURCE_QUALITY_LR_smote_0.5_v1,0.762248,0.856817,0.854527,1.0,keep_baseline
5,20260306T095152Z,Day 04.1,V2_DISPERSION_LR_smote_0.5_v1,0.764527,0.857197,0.864837,1.0,keep_baseline
6,20260306T095152Z,Day 04.1,V2_COMPETITION_LR_smote_0.5_v1,0.763768,0.854539,0.866691,1.0,keep_baseline
7,20260306T095152Z,Day 04.1,V2_DISPERSION_LR_smote_0.5_WITH_DETERMINISTIC_...,0.786935,0.857197,0.864837,1.0,keep_baseline


### Registry post-fix

,run_id,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
0,20260306T_day04_rerun_baseline,Day 02,LR_smote_0.5,0.764527,0.858336,0.865084,1.0,keep_baseline
1,20260306T_day04_rerun_v3a_train,Day 04,V3_A_LR_smote_0.5_v1,0.739081,0.851880,0.853684,1.0,keep_baseline
2,20260306T_day04_rerun_v3a_train,Day 04,V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLL...,0.767186,0.855678,0.853684,1.0,keep_baseline
3,20260306T_day04_rerun_day041_train,Day 04.1,V2_SOURCE_QUALITY_LR_smote_0.5_v1,0.762248,0.857197,0.855956,1.0,keep_baseline
4,20260306T_day04_rerun_day041_train,Day 04.1,V2_DISPERSION_LR_smote_0.5_v1,0.764907,0.857197,0.865057,1.0,keep_baseline
5,20260306T_day04_rerun_day041_train,Day 04.1,V2_COMPETITION_LR_smote_0.5_v1,0.764147,0.852260,0.867425,1.0,keep_baseline
6,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,0.773642,0.860235,0.865847,1.0,keep_baseline
7,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...,0.789594,0.860235,0.865847,1.0,promote
8,20260306T_day04_rerun_day043_train,Day 04.3,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,0.765667,0.862514,0.864081,1.0,keep_baseline
9,20260306T_day04_rerun_day043_train,Day 04.3,V2_DISPERSION_PLUS_TRANSPORT_CARRY30D_LR_smote...,0.761109,0.860235,0.864779,1.0,keep_baseline


## Evidencia del bug fix en extracción

In [4]:
extract_report = json.loads(EXTRACT_REPORT.read_text(encoding='utf-8'))
audit = extract_report['date_resolution_audit']
summary = pd.DataFrame([{
    'autocorrected_files': audit['autocorrected_files'],
    'path_year_to_row_year_2018_2017': audit['path_year_to_row_year'].get('2018->2017', 0),
    'path_year_to_selected_year_2018_2017': audit['path_year_to_selected_year'].get('2018->2017', 0),
    'path_year_to_selected_year_2018_2018': audit['path_year_to_selected_year'].get('2018->2018', 0),
    'path_year_to_selected_year_2021_2020': audit['path_year_to_selected_year'].get('2021->2020', 0),
    'path_year_to_selected_year_2026_2025': audit['path_year_to_selected_year'].get('2026->2025', 0),
}])
display(summary)
display(pd.DataFrame(audit['corrected_files']).head(10))
display(pd.DataFrame(audit['unresolved_conflicts']).head(10))


,autocorrected_files,path_year_to_row_year_2018_2017,path_year_to_selected_year_2018_2017,path_year_to_selected_year_2018_2018,path_year_to_selected_year_2021_2020,path_year_to_selected_year_2026_2025
0,149,147,0,251,0,1


,source_file,fecha_row_candidate,fecha_file_hint,fecha_raw,path_year_hint,date_resolution_status
0,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-02,2022-01-02,2022-01-02,2018,auto_correct_year_from_file_name
1,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-03,2022-01-03,2022-01-03,2018,auto_correct_year_from_file_name
2,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-04,2022-01-04,2022-01-04,2018,auto_correct_year_from_file_name
3,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-05,2022-01-05,2022-01-05,2018,auto_correct_year_from_file_name
4,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-08,2022-01-08,2022-01-08,2018,auto_correct_year_from_file_name
5,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-09,2022-01-09,2022-01-09,2018,auto_correct_year_from_file_name
6,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-10,2022-01-10,2022-01-10,2018,auto_correct_year_from_file_name
7,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-11,2022-01-11,2022-01-11,2018,auto_correct_year_from_file_name
8,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-12,2022-01-12,2022-01-12,2018,auto_correct_year_from_file_name
9,./data/raw/SUPPLIER_DAILY_COMPARISON/2018 COMP...,2021-01-15,2022-01-15,2022-01-15,2018,auto_correct_year_from_file_name


,source_file,fecha_row_candidate,fecha_file_hint,fecha_raw,path_year_hint,date_resolution_status
0,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,2019-11-12,2019-11-13,2019-11-12,2015,row_date_kept_day_month_conflict
1,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-01-27,2020-01-28,2020-01-27,2016,row_date_kept_day_month_conflict
2,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-01-27,2020-01-29,2020-01-27,2016,row_date_kept_day_month_conflict
3,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-02-02,2020-02-03,2020-02-02,2016,row_date_kept_day_month_conflict
4,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-02-23,2020-02-24,2020-02-23,2016,row_date_kept_day_month_conflict
5,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-04-20,2020-04-21,2020-04-20,2016,row_date_kept_day_month_conflict
6,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-05-10,2020-05-11,2020-05-10,2016,row_date_kept_day_month_conflict
7,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-05-24,2020-05-25,2020-05-24,2016,row_date_kept_day_month_conflict
8,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-06-10,2020-06-14,2020-06-10,2016,row_date_kept_day_month_conflict
9,./data/raw/SUPPLIER_DAILY_COMPARISON/2016 COMP...,2020-06-24,2020-06-27,2020-06-24,2016,row_date_kept_day_month_conflict


## Validación de cadena raw -> typed

In [5]:
def mismatch_summary(path: Path, date_col: str) -> pd.DataFrame:
    frame = pd.read_csv(path, usecols=lambda c: c in {'source_file', date_col}, dtype=str, keep_default_na=False)
    years = frame['source_file'].str.extract(r'(20\d{2})(?!.*20\d{2})', expand=False)
    parsed = pd.to_datetime(frame[date_col], errors='coerce').dt.year.astype('Int64')
    mismatch = frame[(years.notna()) & (parsed.notna()) & (years.astype('Int64') != parsed)].copy()
    top_pairs = {}
    if not mismatch.empty:
        vc = mismatch.assign(path_year=years.loc[mismatch.index], parsed_year=parsed.loc[mismatch.index])[['path_year', 'parsed_year']].value_counts().head()
        top_pairs = {f'{a}->{b}': int(v) for (a, b), v in vc.items()}
    return pd.DataFrame([{
        'artifact': str(path.relative_to(PROJECT_ROOT)),
        'mismatch_rows': int(len(mismatch)),
        'unique_files': int(mismatch['source_file'].nunique()) if not mismatch.empty else 0,
        'top_pairs': top_pairs,
    }])

display(pd.concat([
    mismatch_summary(RAW_MINCOST_PATH, 'fecha_raw'),
    mismatch_summary(TYPED_PATH, 'fecha_oferta'),
], ignore_index=True))

v2_df = pd.read_csv(V2_PATH, dtype=str, keep_default_na=False)
v2_target = pd.to_numeric(v2_df['target_elegido'], errors='coerce').fillna(0).astype(int)
positive_events_invalid = int((v2_target.groupby(v2_df['event_id']).sum() != 1).sum())
display(pd.DataFrame([{
    'rows_v2': len(v2_df),
    'events_v2': v2_df['event_id'].nunique(),
    'positive_events_invalid': positive_events_invalid,
}]))


,artifact,mismatch_rows,unique_files,top_pairs
0,data/public/support/ofertas_raw_mincost.csv,960,32,"{'2009->2023': 30, '2010->2020': 30, '2095->20..."
1,data/public/support/ofertas_typed.csv,3469,31,"{'2010->2020': 172, '2098->2022': 166, '2030->..."


,rows_v2,events_v2,positive_events_invalid
0,155946,15404,0


## Baseline corregido

In [6]:
baseline_new = json.loads(BASELINE_NEW.read_text(encoding='utf-8'))
old_baseline = old_registry.loc[old_registry['model_variant'].eq('LR_smote_0.5')].iloc[0]
comparison = pd.DataFrame([
    {
        'version': 'pre_fix_baseline',
        'top1_hit': old_baseline['top1_hit'],
        'top2_hit': old_baseline['top2_hit'],
        'balanced_accuracy': old_baseline['balanced_accuracy'],
        'accuracy': old_baseline['accuracy'],
        'f1_pos': old_baseline['f1_pos'],
        'coverage': old_baseline['coverage'],
        'test_events': old_baseline['test_events'],
    },
    {
        'version': 'post_fix_baseline',
        **baseline_new['metrics'],
    },
])
display(comparison)


,version,top1_hit,top2_hit,balanced_accuracy,accuracy,f1_pos,coverage,test_events
0,pre_fix_baseline,0.772123,0.862894,0.861463,0.909505,0.689893,1.0,2633
1,post_fix_baseline,0.764527,0.858336,0.865084,0.887445,0.652039,1.0,2633


## Day 04 rerun completo

In [7]:
day04_rows = new_registry[new_registry['day_id'].isin(['Day 04', 'Day 04.1', 'Day 04.3'])].copy()
display(day04_rows[cols])


,run_id,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
1,20260306T_day04_rerun_v3a_train,Day 04,V3_A_LR_smote_0.5_v1,0.739081,0.851880,0.853684,1.0,keep_baseline
2,20260306T_day04_rerun_v3a_train,Day 04,V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLL...,0.767186,0.855678,0.853684,1.0,keep_baseline
3,20260306T_day04_rerun_day041_train,Day 04.1,V2_SOURCE_QUALITY_LR_smote_0.5_v1,0.762248,0.857197,0.855956,1.0,keep_baseline
4,20260306T_day04_rerun_day041_train,Day 04.1,V2_DISPERSION_LR_smote_0.5_v1,0.764907,0.857197,0.865057,1.0,keep_baseline
5,20260306T_day04_rerun_day041_train,Day 04.1,V2_COMPETITION_LR_smote_0.5_v1,0.764147,0.852260,0.867425,1.0,keep_baseline
6,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,0.773642,0.860235,0.865847,1.0,keep_baseline
7,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...,0.789594,0.860235,0.865847,1.0,promote
8,20260306T_day04_rerun_day043_train,Day 04.3,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,0.765667,0.862514,0.864081,1.0,keep_baseline
9,20260306T_day04_rerun_day043_train,Day 04.3,V2_DISPERSION_PLUS_TRANSPORT_CARRY30D_LR_smote...,0.761109,0.860235,0.864779,1.0,keep_baseline
10,20260306T_day04_rerun_day043_train,Day 04.3,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_WITH_D...,0.787315,0.862514,0.864081,1.0,keep_baseline


## Day 04.2 y Day 04.3: antes vs después

In [8]:
day042_old = json.loads(DAY042_OLD.read_text(encoding='utf-8'))
day042_new = json.loads(DAY042_NEW.read_text(encoding='utf-8'))
day043_old = json.loads(DAY043_OLD.read_text(encoding='utf-8'))
day043_new = json.loads(DAY043_NEW.read_text(encoding='utf-8'))

compare_042 = pd.DataFrame([
    {'version': 'pre_fix_day042', **day042_old['gate']},
    {'version': 'post_fix_day042', **day042_new['gate']},
])
compare_043 = pd.DataFrame([
    {'version': 'pre_fix_day043', **day043_old['gate']},
    {'version': 'post_fix_day043', **day043_new['gate']},
])
display(Markdown('### Day 04.2 gate'))
display(compare_042)
display(Markdown('### Day 04.3 gate'))
display(compare_043)


### Day 04.2 gate

,version,coverage_gate_pass,leakage_gate_pass,structural_gate_pass,training_allowed,coverage_train_final,coverage_test_final,coverage_train_final_no_lookahead,coverage_test_final_no_lookahead,failure_reasons
0,pre_fix_day042,False,False,True,False,0.780565,0.983743,0.771957,0.983023,"[coverage_below_min_threshold, heuristic_looka..."
1,post_fix_day042,True,False,True,False,0.892423,0.983743,0.885968,0.983023,[heuristic_lookahead_detected]


### Day 04.3 gate

,version,coverage_gate_pass,leakage_gate_pass,structural_gate_pass,training_allowed,coverage_train_final,coverage_test_final,failure_reasons
0,pre_fix_day043,False,True,True,False,0.782978,0.993622,[coverage_below_min_threshold]
1,post_fix_day043,True,True,True,True,0.897175,0.993622,[]


## Decisión final de cierre Day 04

In [9]:
best_pure = new_registry.loc[new_registry['model_variant'].eq('V2_TRANSPORT_ONLY_LR_smote_0.5_v1')]
best_policy = new_registry.loc[new_registry['model_variant'].eq('V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1')]

if not best_pure.empty:
    display(Markdown('### Mejor modelo puro del rerun'))
    display(best_pure[cols])
if not best_policy.empty:
    display(Markdown('### Mejor variante con policy del rerun'))
    display(best_policy[cols])

print('Conclusión operativa:')
print('- El bug de fechas existía y su corrección cambia V2, baseline y la línea de transporte.')
print('- El gate técnico de transporte pasa post-fix en Day 04.1 y Day 04.3.')
print('- Ningún modelo puro supera el gate principal de reentreno frente al baseline corregido.')
print('- La variante V2_TRANSPORT_ONLY + deterministic layer sí pasa el gate de policy y queda como mejor candidato operativo del rerun.')
print('- Day 04 queda cerrado tras el rerun; el siguiente bloque ya es Day 05.')


### Mejor modelo puro del rerun

,run_id,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
6,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,0.773642,0.860235,0.865847,1.0,keep_baseline


### Mejor variante con policy del rerun

,run_id,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
7,20260306T_day04_rerun_day041_train,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...,0.789594,0.860235,0.865847,1.0,promote


Conclusión operativa:
- El bug de fechas existía y su corrección cambia V2, baseline y la línea de transporte.
- El gate técnico de transporte pasa post-fix en Day 04.1 y Day 04.3.
- Ningún modelo puro supera el gate principal de reentreno frente al baseline corregido.
- La variante V2_TRANSPORT_ONLY + deterministic layer sí pasa el gate de policy y queda como mejor candidato operativo del rerun.
- Day 04 queda cerrado tras el rerun; el siguiente bloque ya es Day 05.


## Decisión final del rerun Day 04

El bug de fechas existía de verdad y afectaba al entrenamiento histórico porque contaminaba la cadena `raw -> typed -> V2`. Por eso el baseline oficial tuvo que rehacerse: no cambió la receta del modelo, cambió la base de datos sobre la que estaba entrenado. El baseline corregido pasa a ser la referencia vigente del proyecto y sustituye al baseline anterior, que queda archivado solo como evidencia pre-fix.

Con el rerun post-fix ya sí pudimos entrenar variantes que antes estaban bloqueadas, especialmente `transport_only`. Aun así, ningún modelo puro superó el gate principal frente al baseline corregido. Sí hubo mejoras parciales:
- `V2_COMPETITION_LR_smote_0.5_v1` mejora `balanced_accuracy`, pero empeora `Top-2`.
- `V2_TRANSPORT_ONLY_LR_smote_0.5_v1` mejora ligeramente `Top-2` y `balanced_accuracy`, pero no lo suficiente para promoción.
- `V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1` consigue el mejor `Top-2` entre los modelos puros del rerun, pero empeora `balanced_accuracy`.
- `V2_DISPERSION_PLUS_TRANSPORT_CARRY30D_LR_smote_0.5_v1` también mejora algo `Top-2`, pero vuelve a perder en `balanced_accuracy`.

La mejor decisión operativa del rerun no viene de un modelo puro nuevo, sino de combinar `V2_TRANSPORT_ONLY_LR_smote_0.5` con la capa determinista Day 03. Esa variante sí pasa el gate de policy: mejora `Top-1`, mantiene `Top-2`, sube la coherencia a `1.0` y no introduce overrides dañinos. Por eso queda como mejor candidato operativo del bloque Day 04, mientras que el champion de modelo sigue siendo el baseline corregido.

Conclusión senior de Day 04:
- champion de modelo = `LR_smote_0.5` sobre `V2` corregido;
- mejor resultado operativo = `V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1`;
- Day 04 queda cerrado tras el rerun post-fix;
- el siguiente bloque correcto ya es Day 05.